# 06 - Reconcile control and committed output

Detects publication, lease, event, attempt, and count inconsistencies. Findings use stable IDs so recurring checks update one open finding and resolved conditions retain history.

**After importing into Fabric:** On the configuration code cell, select **... -> Toggle parameter cell** and confirm the parameter indicator. Then attach and pin `people_counter_<environment>` as this notebook's default Lakehouse.

In [ ]:
DATABASE = ""
TABLE_PREFIX = "people_counter"

In [ ]:
from datetime import datetime, timezone
import json
import re

from people_counter.fabric_control import ControlWriter
import notebookutils
from pyspark.sql import DataFrame, SparkSession, functions as F


IDENTIFIER = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*$")
database = DATABASE.strip()
prefix = TABLE_PREFIX.strip()
if database and IDENTIFIER.fullmatch(database) is None:
    raise ValueError("DATABASE is not a valid identifier")
if IDENTIFIER.fullmatch(prefix) is None:
    raise ValueError("TABLE_PREFIX is not a valid identifier")


def table(suffix: str) -> str:
    value = f"{prefix}_{suffix}"
    return f"{database}.{value}" if database else value


spark_candidate = globals().get("spark")
if not isinstance(spark_candidate, SparkSession):
    raise RuntimeError("A Fabric Spark session is required")
spark_session = spark_candidate
spark_session.conf.set("spark.sql.session.timeZone", "UTC")
writer = ControlWriter(spark_session, table("control_writer"))
DeltaTable = writer.tables
now = datetime.now(timezone.utc)
work = spark_session.table(table("video_work"))
attempts = spark_session.table(table("video_attempts"))
receipts = spark_session.table(table("event_receipts"))
line_counts = spark_session.table(table("line_count_attempts"))
findings_table = table("reconciliation_findings")


def finding_frame(
    source: DataFrame,
    severity: str,
    finding_type: str,
    detail_columns: list[str],
    identity_columns: list[str] | None = None,
) -> DataFrame:
    work_key = F.coalesce(F.col("work_id"), F.lit(""))
    attempt_key = F.coalesce(F.col("attempt_id"), F.lit(""))
    identity_values = [
        F.coalesce(F.col(column).cast("string"), F.lit(""))
        for column in (identity_columns or [])
    ]
    return source.select(
        F.sha2(
            F.concat_ws("\n", F.lit(finding_type), work_key, attempt_key, *identity_values),
            256,
        ).alias("finding_id"),
        F.lit(now).alias("detected_at"),
        F.lit(now).alias("last_detected_at"),
        F.lit(severity).alias("severity"),
        F.lit(finding_type).alias("finding_type"),
        F.col("work_id"),
        F.col("attempt_id"),
        F.to_json(F.struct(*[F.col(column) for column in detail_columns])).alias("details"),
        F.lit(None).cast("timestamp").alias("resolved_at"),
        F.col("capture_date"),
    )


missing_committed_attempt = (
    work.where((F.col("status") == "SUCCEEDED") & F.col("committed_attempt_id").isNotNull())
    .alias("w")
    .join(
        attempts.select(
            F.col("attempt_id").alias("found_attempt_id"),
            F.col("work_id").alias("found_work_id"),
            F.col("capture_date").alias("found_capture_date"),
            F.col("status").alias("found_attempt_status"),
            F.col("config_sha256").alias("found_config_sha256"),
        ).alias("a"),
        (F.col("w.committed_attempt_id") == F.col("a.found_attempt_id"))
        & (F.col("w.work_id") == F.col("a.found_work_id"))
        & (F.col("w.capture_date") == F.col("a.found_capture_date")),
        "left",
    )
    .where(
        F.col("found_attempt_id").isNull()
        | (F.col("found_attempt_status") != "SUCCEEDED")
        | ~F.col("a.found_config_sha256").eqNullSafe(F.col("w.config_sha256"))
    )
    .select(
        F.col("w.work_id"),
        F.col("w.committed_attempt_id").alias("attempt_id"),
        F.col("w.status"),
        F.col("a.found_attempt_status"),
        F.col("w.config_sha256").alias("work_config_sha256"),
        F.col("a.found_config_sha256"),
        F.col("w.capture_date"),
    )
)

pointer_on_non_success = work.where(
    (F.col("status") != "SUCCEEDED") & F.col("committed_attempt_id").isNotNull()
).select(
    "work_id",
    F.col("committed_attempt_id").alias("attempt_id"),
    "status",
    "capture_date",
)

success_without_pointer = work.where(
    (F.col("status") == "SUCCEEDED") & F.col("committed_attempt_id").isNull()
).select(
    "work_id",
    F.lit(None).cast("string").alias("attempt_id"),
    "status",
    "capture_date",
)

orphan_success = (
    attempts.where(F.col("status") == "SUCCEEDED")
    .alias("a")
    .join(work.select("work_id", "committed_attempt_id").alias("w"), "work_id", "left")
    .where(F.col("w.committed_attempt_id").isNull() | (F.col("w.committed_attempt_id") != F.col("a.attempt_id")))
    .select(
        F.col("a.work_id"),
        F.col("a.attempt_id"),
        F.col("a.status"),
        F.col("w.committed_attempt_id"),
        F.col("a.capture_date"),
    )
)

receipt_without_work = (
    receipts.where(F.col("registration_status").isin("QUEUED", "EXISTING_WORK"))
    .alias("r")
    .join(work.select(F.col("work_id").alias("found_work_id")).alias("w"), F.col("r.work_id") == F.col("w.found_work_id"), "left")
    .where(F.col("found_work_id").isNull())
    .select(
        F.col("r.work_id"),
        F.lit(None).cast("string").alias("attempt_id"),
        "event_key",
        "registration_status",
        F.lit(None).cast("date").alias("capture_date"),
    )
)

expired_active = work.where(
    F.col("status").isin("LEASED", "STAGING", "RUNNING", "WRITING")
    & (F.col("lease_expires_at") < F.lit(now))
).select(
    "work_id",
    F.col("lease_owner_attempt_id").alias("attempt_id"),
    "status",
    "lease_expires_at",
    "capture_date",
)

line_final = line_counts.groupBy("work_id", "attempt_id", "capture_date").agg(
    F.max("cumulative_in_count").alias("detail_in"),
    F.max("cumulative_out_count").alias("detail_out"),
)
count_mismatch = (
    attempts.where(F.col("status") == "SUCCEEDED")
    .select("work_id", "attempt_id", "line_in_count", "line_out_count", "capture_date")
    .join(line_final, ["work_id", "attempt_id", "capture_date"], "inner")
    .where((F.col("line_in_count") != F.col("detail_in")) | (F.col("line_out_count") != F.col("detail_out")))
)

duplicate_work_identity = (
    work.groupBy("source_uri", "asset_version")
    .agg(
        F.count("work_id").alias("duplicate_count"),
        F.min("capture_date").alias("capture_date"),
    )
    .where(F.col("duplicate_count") > 1)
    .select(
        F.sha2(F.concat_ws("\n", "source_uri", "asset_version"), 256).alias("work_id"),
        F.lit(None).cast("string").alias("attempt_id"),
        "source_uri",
        "asset_version",
        "duplicate_count",
        "capture_date",
    )
)

committed_attempts = (
    work.where(F.col("status") == "SUCCEEDED")
    .select("work_id", F.col("committed_attempt_id").alias("attempt_id"), "capture_date")
    .join(
        attempts.select("work_id", "attempt_id", "capture_date", "processed_frames"),
        ["work_id", "attempt_id", "capture_date"],
    )
    .where(F.col("processed_frames") > 0)
)
committed_line_keys = line_counts.select("work_id", "attempt_id", "capture_date").dropDuplicates()
missing_line_output = (
    committed_attempts.alias("a")
    .join(committed_line_keys.alias("l"), ["work_id", "attempt_id", "capture_date"], "left_anti")
    .select("work_id", "attempt_id", "processed_frames", "capture_date")
)

frames = [
    finding_frame(
        missing_committed_attempt,
        "ERROR",
        "INVALID_COMMITTED_ATTEMPT",
        ["status", "found_attempt_status", "work_config_sha256", "found_config_sha256"],
    ),
    finding_frame(success_without_pointer, "ERROR", "SUCCESS_WITHOUT_COMMIT_POINTER", ["status"]),
    finding_frame(pointer_on_non_success, "ERROR", "COMMIT_POINTER_ON_NON_SUCCESS", ["status"]),
    finding_frame(orphan_success, "WARN", "UNCOMMITTED_SUCCESS_ATTEMPT", ["status", "committed_attempt_id"]),
    finding_frame(
        receipt_without_work,
        "ERROR",
        "RECEIPT_WITHOUT_WORK",
        ["event_key", "registration_status"],
        identity_columns=["event_key"],
    ),
    finding_frame(expired_active, "ERROR", "EXPIRED_ACTIVE_LEASE", ["status", "lease_expires_at"]),
    finding_frame(count_mismatch, "ERROR", "FINAL_LINE_COUNT_MISMATCH", ["line_in_count", "detail_in", "line_out_count", "detail_out"]),
    finding_frame(duplicate_work_identity, "ERROR", "DUPLICATE_WORK_IDENTITY", ["source_uri", "asset_version", "duplicate_count"]),
    finding_frame(missing_line_output, "ERROR", "COMMITTED_WITHOUT_LINE_OUTPUT", ["processed_frames"]),
]
current = frames[0]
for frame in frames[1:]:
    current = current.unionByName(frame)

duplicate_current_findings = current.groupBy("finding_id").count().where(F.col("count") != 1)
if duplicate_current_findings.head(1):
    raise RuntimeError("Reconciliation produced duplicate finding_id values")

target = DeltaTable.forName(spark_session, findings_table)
(
    target.alias("t")
    .merge(current.alias("s"), "t.finding_id = s.finding_id")
    .whenMatchedUpdate(
        set={
            "last_detected_at": "s.last_detected_at",
            "severity": "s.severity",
            "details": "s.details",
            "resolved_at": "NULL",
        }
    )
    .whenNotMatchedInsertAll()
    .execute()
)
target.update(
    condition=F.col("resolved_at").isNull() & (F.col("last_detected_at") < F.lit(now)),
    set={"resolved_at": F.lit(now)},
)
current_ids = current.select("finding_id").distinct()
persisted_current = (
    spark_session.table(findings_table)
    .join(current_ids, "finding_id", "inner")
    .where(F.col("resolved_at").isNull())
)
persisted_counts = persisted_current.groupBy("finding_id").count()
if persisted_counts.where(F.col("count") != 1).head(1):
    raise RuntimeError("Reconciliation findings are not unique after merge")
if persisted_counts.count() != current_ids.count():
    raise RuntimeError("Not every current reconciliation finding persisted unresolved")
counts = {row.severity: row["count"] for row in current.groupBy("severity").count().collect()}
outcome = {"checked_at": now.isoformat(), "open_findings": sum(counts.values()), "by_severity": counts}
print(json.dumps(outcome, sort_keys=True))
if counts.get("ERROR", 0):
    print("ERROR findings are open; inspect the reconciliation table and operations report")

In [ ]:
notebookutils.notebook.exit(json.dumps(outcome, sort_keys=True))